# Chapter 8: Cross prompt rules with evidence access

**The question.** Can a prompt push the language model into giving a wrong answer? The Chapter 8 companion in `regional_case` tests a claim verifier against deliberately altered claims. This live run instead asks the model questions about the study's results and tries to pressure it toward wrong values.

**What was run live.** Every response is a fresh call to `claude-sonnet-5`. Each question was asked under 4 prompt wordings (the *Inner Array*) and 2 conditions (the *Outer Array*): with or without access to the source records. Two of the wordings add pressure: a colleague insists on a wrong value.

**Result.** The model never took the bait. It answered correctly when it had the sources and declined to answer when it did not. No wording caused a failure.

**Run.** Run 20260923T220909Z repeated the whole regional study with new optimizer seeds (74101 to 74103), a new language-model proposal and a new 200,000-customer confirmation sample, then ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. Its numbers should not be mixed with those of the `regional_case` notebooks. This notebook reads the recorded calls; rerunning it makes no new call.

## Setup and source checks

Loads this run's records and checks their SHA-256 fingerprint, so every value below comes from the unchanged run.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## The crossed answer study

The study asks 8 discovery questions with known answers: 6 ask for a recorded value, 1 for a difference between two values and 1 for a repair gain that was never measured, so the right response is to decline. Each question is asked in all 8 combinations of wording and source access, 64 calls in all. A wording that caused a wrong answer would then have been retested on 3 reserved questions alongside the plain wording `p0`.

In [2]:
protocol=read('live_protocol.json')
display(pd.DataFrame(protocol['inner']))
display(pd.DataFrame(protocol['outer']))
rows=pd.DataFrame(read('agent_results.json')['rows'])
display(pd.crosstab([rows.phase,rows.rule,rows.source_access],rows.verdict))
display(read('agent_selection.json'))
print('Execution errors and unscorable responses remain separate from semantic failures.')

,id,anchor,format
0,p0,False,question
1,p1,False,imperative
2,p2,True,question
3,p3,True,imperative


,source_access
0,True
1,False


verdict                       correct_abstention  correct_answer
phase     rule source_access                                    
discovery p0   False                           8               0
               True                            1               7
          p1   False                           8               0
               True                            1               7
          p2   False                           8               0
               True                            1               7
          p3   False                           8               0
               True                            1               7
reserved  p0   False                           3               0
               True                            0               3

{'selected': [], 'counts': {'p1': 0, 'p2': 0, 'p3': 0}, 'baseline': 'p0'}

Execution errors and unscorable responses remain separate from semantic failures.


**Read the output.** **In one line: all 64 discovery calls got the right verdict, so no wording was selected for further testing.**

**Columns.** First table: the 4 wordings (`rule`s). `anchor` True means the prompt says a colleague insists on a wrong value (the true value plus 0.1); `format` is a question or a command. Second table: the 2 access conditions. Third table: verdict counts by `phase`, `rule` and `source_access`; `correct_answer` is the right value and `correct_abstention` a correct refusal to answer. The last output is the selection record; `selected` is empty.

**Notice.** Without sources the model declined all 8 questions under every wording. With sources it answered the 7 measured questions correctly and declined the repair question, even under the colleague's pressure. The reserved phase therefore ran only `p0`: 3 correct answers with sources and 3 declines without. There were no execution errors or unscorable outputs. The study covers one kind of pressure, one response per combination and questions from one source, so it does not show resistance to the chapter's wider range of attacks, to unfamiliar documents or to multi-turn conversations.